# 📊 Importar KM Dinâmico - Multi Anos (2023-2026)

Script que:
- ✅ Processa anos 2023, 2024, 2025, 2026
- ✅ Cria tabela separada por ano
- ✅ Remove duplicados do dataframe
- ✅ Verifica registos já existentes
- ✅ Insere APENAS registos novos
- ✅ Usa UNIQUE constraint para evitar duplicados

In [1]:
import os
import pandas as pd
import openpyxl
from datetime import datetime
import sqlite3
import platform
import re

print("✅ Imports carregados")

✅ Imports carregados


In [2]:
# ── PARAMETRO: QUAL ANO IMPORTAR? ────────────────────────────────────────
# 👇 ALTERE PARA O ANO QUE PRETENDE: 2023, 2024, 2025 ou 2026
ANO_IMPORTACAO = 2026

print(f"📅 ANO A IMPORTAR: {ANO_IMPORTACAO}")

📅 ANO A IMPORTAR: 2026


In [3]:
# ── CONFIGURAÇÕES ─────────────────────────────────────────────────────────
ANOS = [ANO_IMPORTACAO]
BASE_PATH_ROOT = r"T:\Portugal\D-Trafico\KM BASE"

MONTH_FILES = {
    1:  "01 JANEIRO.xlsx",
    2:  "02 FEVEREIRO.xlsx",
    3:  "03 MARÇO.xlsx",
    4:  "04 ABRIL.xlsx",
    5:  "05 MAIO.xlsx",
    6:  "06 JUNHO.xlsx",
    7:  "07 JULHO.xlsx",
    8:  "08 AGOSTO.xlsx",
    9:  "09 Setembro.xlsx",
    10: "10 Outubro.xlsx",
    11: "11 Novembro.xlsx",
    12: "12 Dezembro.xlsx",
}

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    DB_PATH = "inform_27.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"📂 Base path: {BASE_PATH_ROOT}")
print(f"📅 Anos a processar: {ANOS}")
print(f"📋 Meses por ano: {len(MONTH_FILES)}")
print(f"💾 BD: {DB_PATH}")

📂 Base path: T:\Portugal\D-Trafico\KM BASE
📅 Anos a processar: [2026]
📋 Meses por ano: 12
💾 BD: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [4]:
# ── FUNÇÕES DE PARSING ────────────────────────────────────────────────────
def clean_value(val):
    """Limpar e converter valores."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return 0
    if isinstance(val, (int, float)):
        return float(val)
    return 0

def parse_formula(formula_str):
    """Extrair tipo_contrato, km_gratis e rate_km da fórmula."""
    if not formula_str:
        return None, 0, 0
    
    tipo_formula = "desconhecido"
    km_gratis = 0
    rate = 0
    
    if "fixo" in str(formula_str).lower():
        tipo_formula = "fixo"
    elif "km" in str(formula_str).lower():
        tipo_formula = "km"
    
    if "-" in str(formula_str):
        try:
            parts = str(formula_str).split("-")
            if len(parts) >= 2:
                km_gratis = float(parts[1].replace(")", "").strip())
        except:
            pass
    
    return tipo_formula, km_gratis, rate

def parse_cell_a(cell_value, sheet_name, row_idx, anon_map):
    """Trator = XX-YY-ZZ (2-2-2 exatamente) ou AABBBB
    Reboque = tudo o resto
    Descrição = tipo, nomes"""
    if not cell_value:
        return sheet_name, None, None, None
    
    cell_str = str(cell_value).strip()
    transportador = sheet_name.strip()
    
    trator = None
    reboque = None
    
    matricula_std = r'([A-Z0-9]{2}-[A-Z0-9]{2}-[A-Z0-9]{2})'
    codigo_trator = r'(?<![A-Z0-9-])([A-Z]{2}\d{4,5})(?![A-Z0-9-])'
    
    tratores_candidatos = []
    
    for match in re.finditer(matricula_std, cell_str):
        m_str = match.group(1)
        partes = m_str.split('-')
        if len(partes) == 3 and len(partes[0]) == 2 and len(partes[1]) == 2 and len(partes[2]) == 2:
            tratores_candidatos.append((m_str, match.start(), 'std'))
    
    for match in re.finditer(codigo_trator, cell_str):
        tratores_candidatos.append((match.group(1), match.start(), 'cod'))
    
    tratores_candidatos.sort(key=lambda x: x[1])
    
    if tratores_candidatos:
        trator = tratores_candidatos[0][0]
    
    if len(tratores_candidatos) >= 2:
        reboque = tratores_candidatos[1][0]
    else:
        temp_str = cell_str
        if trator:
            temp_str = temp_str.replace(trator, '', 1)
        
        match_l = re.search(r'(L-?\d+)', temp_str)
        if match_l:
            reboque = match_l.group(1)
        else:
            match_cod = re.search(r'([A-Z]{1,2}-\d{4,5})', temp_str)
            if match_cod:
                reboque = match_cod.group(1)
    
    descricao = cell_str
    if trator:
        descricao = descricao.replace(trator, '', 1)
    if reboque:
        descricao = descricao.replace(reboque, '', 1)
    
    parts = descricao.split()
    if parts and len(parts[0]) < 10 and parts[0].isalpha():
        if len(parts[0]) < 6:
            descricao = ' '.join(parts[1:])
    
    descricao = re.sub(r'\s*\/\s*', ' ', descricao)
    descricao = re.sub(r'\(\s*\)', '', descricao)
    descricao = re.sub(r'\s+', ' ', descricao)
    descricao = descricao.strip()
    
    return transportador, descricao if descricao else None, trator, reboque


def extract_sheet_data(ws_values, ws_formulas, sheet_name, month):
    """Extrair dados da folha de cálculo."""
    sheet_data = []
    
    first_row = list(ws_values.iter_rows(min_row=1, max_row=1, values_only=True))[0]
    date_columns = {}
    
    for col_idx in range(2, len(first_row)):
        val = first_row[col_idx]
        if isinstance(val, datetime):
            date_columns[col_idx] = val
    
    if not date_columns:
        return sheet_data
    
    anon_map = {}
    row_idx = 2
    
    while row_idx <= ws_values.max_row:
        current_row = list(ws_values.iter_rows(
            min_row=row_idx, max_row=row_idx, values_only=True))[0]
        
        if not current_row or not current_row[0]:
            row_idx += 1
            continue
        
        col_b = str(current_row[1]).strip().upper() if current_row[1] else ""
        if col_b != "KM":
            row_idx += 1
            continue
        
        transportador, descricao, trator, reboque = parse_cell_a(
            current_row[0], sheet_name, row_idx, anon_map)
        
        # Usar trator ou reboque para determinar se é válido
        if not (trator or reboque):
            row_idx += 1
            continue
        
        km_row = current_row
        p_row = list(ws_values.iter_rows(min_row=row_idx+1, max_row=row_idx+1, values_only=True))[0]
        vt_row = list(ws_values.iter_rows(min_row=row_idx+2, max_row=row_idx+2, values_only=True))[0]
        sd_row = list(ws_values.iter_rows(min_row=row_idx+3, max_row=row_idx+3, values_only=True))[0]
        tot_row = list(ws_values.iter_rows(min_row=row_idx+4, max_row=row_idx+4, values_only=True))[0]
        
        formula_cell = ws_formulas.cell(row=row_idx + 3, column=3).value
        tipo_formula, km_gratis, rate = parse_formula(formula_cell)
        
        for col_idx, date_obj in date_columns.items():
            if col_idx >= len(km_row):
                continue
            
            km_value = clean_value(km_row[col_idx])
            if not km_value or km_value == 0:
                continue
            
            portagens = clean_value(p_row[col_idx] if col_idx < len(p_row) else None)
            valor_total = clean_value(vt_row[col_idx] if col_idx < len(vt_row) else None)
            sub_divisao = clean_value(sd_row[col_idx] if col_idx < len(sd_row) else None)
            preco_base = valor_total - sub_divisao if (valor_total and sub_divisao) else valor_total
            
            sheet_data.append({
                'transportador': transportador,
                'descricao': descricao,
                'trator': trator,
                'reboque': reboque,
                'data': date_obj,
                'dia': date_obj.day,
                'mes': month,
                'km': int(km_value) if km_value else 0,
                'portagens': portagens,
                'preco_base': preco_base,
                'sub_divisao': sub_divisao,
                'tipo_contrato': tipo_formula,
                'km_gratis': km_gratis,
                'rate_km': rate,
                'total': valor_total,
            })
        
        row_idx += 5
    
    return sheet_data

print("✅ Funções carregadas")

✅ Funções carregadas


In [5]:
# ── PROCESSAR CADA ANO ────────────────────────────────────────────────────
for ano in ANOS:
    print(f"\n{'='*70}")
    print(f"📅 PROCESSANDO ANO {ano}")
    print(f"{'='*70}\n")
    
    BASE_PATH = rf"{BASE_PATH_ROOT}\Ano {ano}"
    table_name = f"km_diario_{ano}"
    
    if not os.path.exists(BASE_PATH):
        print(f"   ⚠️  Pasta não encontrada: {BASE_PATH}\n")
        continue
    
    # Criar tabela (se não existir)
    print(f"🔨 Tabela '{table_name}'...\n")
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            transportador   TEXT NOT NULL,
            descricao       TEXT,
            tipo_veiculo          TEXT,
            viatura         TEXT,
            data            DATE NOT NULL,
            dia             INTEGER,
            mes             INTEGER,
            km              INTEGER,
            portagens       REAL,
            preco_base      REAL,
            sub_divisao     REAL,
            tipo_contrato   TEXT,
            km_gratis       REAL,
            rate_km         REAL,
            total           REAL,
            criado_em       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(transportador, tipo_veiculo, viatura, data)
        )
    """)
    conn.commit()
    
    # Ler ficheiros do ano
    all_data = []
    
    for month, filename in MONTH_FILES.items():
        filepath = os.path.join(BASE_PATH, filename)
        
        if not os.path.exists(filepath):
            continue
        
        try:
            wb_val = openpyxl.load_workbook(filepath, data_only=True)
            wb_form = openpyxl.load_workbook(filepath, data_only=False)
            
            data = []
            for sheet_name in wb_val.sheetnames:
                ws_v = wb_val[sheet_name]
                ws_f = wb_form[sheet_name]
                sheet_data = extract_sheet_data(ws_v, ws_f, sheet_name.strip(), month)
                data.extend(sheet_data)
            
            all_data.extend(data)
            wb_val.close()
            wb_form.close()
            print(f"   ✅ {filename}: {len(data)} registos")
        except Exception as e:
            print(f"   ❌ {filename}: {e}")
    
    if not all_data:
        print(f"   ⚠️  Nenhum dado encontrado para {ano}\n")
        continue
    
    df = pd.DataFrame(all_data)
    
    # Mapear colunas para nomes da BD
    df = df.rename(columns={
        'trator': 'tipo_veiculo',
        'reboque': 'viatura'
    })
    
    # Remover duplicados do dataframe
    print(f"\n🔍 Removendo duplicados do dataframe...")
    df_before = len(df)
    df = df.drop_duplicates(
        subset=['transportador', 'tipo_veiculo', 'viatura', 'data'], 
        keep='first'
    )
    df_removed = df_before - len(df)
    
    if df_removed > 0:
        print(f"   ⚠️  {df_removed} registos removidos")
    else:
        print(f"   ✅ Nenhum duplicado")
    
    # Verificar registos já existentes
    print(f"\n🔍 Verificando registos já existentes...")
    existing = cursor.execute(f"""
        SELECT transportador, tipo_veiculo, viatura, DATE(data) FROM {table_name}
    """).fetchall()
    
    existing_set = {(t[0], t[1], t[2], t[3]) for t in existing}
    print(f"   📊 Já existem: {len(existing_set):,}")
    
    # Filtrar novos registos
    df['data_str'] = df['data'].dt.strftime('%Y-%m-%d')
    df['_key'] = df.apply(
        lambda x: (x['transportador'], x['tipo_veiculo'], x['viatura'], x['data_str']), 
        axis=1
    )
    
    df_insert = df[~df['_key'].isin(existing_set)].copy()
    df_insert = df_insert.drop(['data_str', '_key'], axis=1)
    
    print(f"   ➡️  Novos registos: {len(df_insert):,}")
    print(f"   ⏭️  Já na BD: {len(df) - len(df_insert):,}")
    
    # Inserir
    if len(df_insert) > 0:
        try:
            df_insert.to_sql(
                name=table_name,
                con=conn,
                if_exists='append',
                index=False,
            )
            conn.commit()
            print(f"   ✅ Inserção concluída")
        except Exception as e:
            print(f"   ❌ Erro: {e}")
            conn.rollback()
    else:
        print(f"   ⏭️  Nenhum novo registro para inserir")
    
    # Resumo
    total = cursor.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"\n   📊 Total na tabela '{table_name}': {total:,}\n")

conn.close()
print(f"{'='*70}")
print(f"✅ Processo concluído com sucesso!")
print(f"{'='*70}")


📅 PROCESSANDO ANO 2026

🔨 Tabela 'km_diario_2026'...

   ✅ 01 JANEIRO.xlsx: 524 registos
   ✅ 02 FEVEREIRO.xlsx: 417 registos
   ✅ 03 MARÇO.xlsx: 562 registos
   ✅ 04 ABRIL.xlsx: 575 registos
   ✅ 05 MAIO.xlsx: 535 registos
   ✅ 06 JUNHO.xlsx: 560 registos
   ✅ 07 JULHO.xlsx: 589 registos
   ✅ 08 AGOSTO.xlsx: 503 registos
   ✅ 09 Setembro.xlsx: 0 registos

🔍 Removendo duplicados do dataframe...
   ✅ Nenhum duplicado

🔍 Verificando registos já existentes...
   📊 Já existem: 3,799
   ➡️  Novos registos: 4,265
   ⏭️  Já na BD: 0
   ❌ Erro: Execution failed

   📊 Total na tabela 'km_diario_2026': 3,799

✅ Processo concluído com sucesso!
